# DnCNN-S training reproduction (Kaggle)
Reproduces DnCNN-S (sigma=25) from *Beyond a Gaussian Denoiser* (Zhang et al., 2017).
Residual learning + batch normalization, Train400, 40x40 patches, Adam.

**Before running:** Settings (right panel) -> Accelerator -> **GPU T4 x2** (or P100),
and add your uploaded data via **+ Add Input**.

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Locate the data folders (auto-detect under /kaggle/input)

In [ ]:
import os, glob
def find_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    hits = [h for h in hits if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f'Could not find a folder named {name} under /kaggle/input. '
                                f'Did you add the dataset via + Add Input?')
    return hits[0]
TRAIN_DIR = find_dir('Train400')
SET12_DIR = find_dir('set12')
BSD68_DIR = find_dir('bsd68')
print('Train400:', TRAIN_DIR, '->', len(glob.glob(TRAIN_DIR+'/*.png')), 'png')
print('set12:   ', SET12_DIR, '->', len(glob.glob(SET12_DIR+'/*.png')), 'png')
print('bsd68:   ', BSD68_DIR, '->', len(glob.glob(BSD68_DIR+'/*.png')), 'png')

## 2. Model (DnCNN, 17 layers, predicts the residual noise)

In [ ]:
import torch.nn as nn
class DnCNN(nn.Module):
    def __init__(self, depth=17, n_channels=64, image_channels=1, ks=3):
        super().__init__()
        pad = ks // 2
        layers = [nn.Conv2d(image_channels, n_channels, ks, padding=pad, bias=True), nn.ReLU(True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(n_channels, n_channels, ks, padding=pad, bias=False),
                       nn.BatchNorm2d(n_channels, eps=1e-4, momentum=0.95), nn.ReLU(True)]
        layers += [nn.Conv2d(n_channels, image_channels, ks, padding=pad, bias=False)]
        self.dncnn = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.dncnn(x)
m = DnCNN()
print('params:', sum(p.numel() for p in m.parameters()))

## 3. Dataset (random 40x40 patches, 8-mode augmentation, AWGN sigma=25)

In [ ]:
import numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
BATCH, PS, SIGMA = 128, 40, 25
def data_aug(img, mode):
    return [img, np.flipud(img), np.rot90(img), np.flipud(np.rot90(img)),
            np.rot90(img,2), np.flipud(np.rot90(img,2)),
            np.rot90(img,3), np.flipud(np.rot90(img,3))][mode]
class DenoisingDataset(Dataset):
    def __init__(self, image_dir, ps=PS, sigma=SIGMA, steps=1600*BATCH):
        self.images = [cv2.imread(p, cv2.IMREAD_GRAYSCALE)
                       for p in sorted(glob.glob(image_dir+'/*.png'))]
        self.ps, self.sigma, self.steps = ps, sigma, steps
    def __len__(self): return self.steps
    def __getitem__(self, idx):
        img = self.images[np.random.randint(len(self.images))]
        H, W = img.shape; ps = self.ps
        i = np.random.randint(0, H-ps+1); j = np.random.randint(0, W-ps+1)
        patch = data_aug(img[i:i+ps, j:j+ps], np.random.randint(0,8)).astype(np.float32)/255.
        noise = np.random.normal(0, self.sigma/255., patch.shape).astype(np.float32)
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).unsqueeze(0)
        return to_t(patch+noise), to_t(noise)

## 4. Evaluation (PSNR on a test set, fixed-seed noise)

In [ ]:
@torch.no_grad()
def evaluate(model, test_dir, sigma=SIGMA):
    model.eval(); psnrs = []
    for p in sorted(glob.glob(test_dir+'/*.png')):
        clean = cv2.imread(p, cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.
        np.random.seed(0)
        noisy = clean + np.random.normal(0, sigma/255., clean.shape).astype(np.float32)
        x = torch.from_numpy(noisy).view(1,1,*clean.shape).float().to(device)
        den = (x - model(x)).clamp(0,1).cpu().numpy().squeeze()
        psnrs.append(10*np.log10(1.0/np.mean((den-clean)**2)))
    return float(np.mean(psnrs))

## 5. Train
50 epochs to match the paper. Kaggle sessions last up to ~12 h, so this finishes in one run.
Best checkpoint (by Set12 PSNR) is saved to `dncnn_s25_best.pth`.

In [ ]:
import time
EPOCHS = 50
LR = 1e-3
QUICK = False                       # set True for a fast 3-epoch sanity check
steps = 1600*BATCH
if QUICK: EPOCHS, steps = 3, 200*BATCH

model = DnCNN(depth=17).to(device)
criterion = nn.MSELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
milestones = [m for m in [30, 45] if m < EPOCHS]
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)

loader = DataLoader(DenoisingDataset(TRAIN_DIR, steps=steps),
                    batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
best = 0.0
for epoch in range(1, EPOCHS+1):
    model.train(); t0 = time.time(); running = 0.0
    for noisy, noise in loader:
        noisy, noise = noisy.to(device), noise.to(device)
        optimizer.zero_grad()
        loss = criterion(model(noisy), noise) / (2*noisy.size(0))
        loss.backward(); optimizer.step()
        running += loss.item()
    scheduler.step()
    ps12 = evaluate(model, SET12_DIR)
    if ps12 > best:
        best = ps12; torch.save(model.state_dict(), 'dncnn_s25_best.pth')
    print(f'epoch {epoch:2d}/{EPOCHS} | loss {running/len(loader):.4f} '
          f'| Set12 {ps12:.2f} dB | best {best:.2f} | lr {optimizer.param_groups[0]["lr"]:.1e} '
          f'| {time.time()-t0:.0f}s')
print('Training done. Best Set12 PSNR:', round(best,2))

## 6. Final evaluation (best checkpoint) vs. paper

In [ ]:
model.load_state_dict(torch.load('dncnn_s25_best.pth'))
s12 = evaluate(model, SET12_DIR); b68 = evaluate(model, BSD68_DIR)
print(f'Set12 PSNR: {s12:.2f} dB   (paper 30.44)')
print(f'BSD68 PSNR: {b68:.2f} dB   (paper 29.23)')

## 7. Download the trained model
The file `dncnn_s25_best.pth` appears in the right-panel **Output** tab once the
session is committed/saved -> download it from there (Kaggle wipes the runtime otherwise).